In [18]:
import pandas as pd

df = pd.read_csv('data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            248 non-null    str    
 1   selic           248 non-null    float64
 2   ipca            248 non-null    float64
 3   ibc_br          248 non-null    float64
 4   preco_gasolina  248 non-null    float64
dtypes: float64(4), str(1)
memory usage: 9.8 KB


In [19]:
df.head(5)

,Date,selic,ipca,ibc_br,preco_gasolina
0,2005-05-31,19.60,0.49,178129.5,1.814208
1,2005-06-30,19.75,-0.02,179509.8,1.775512
2,2005-07-31,19.75,0.25,181673.2,1.785572
3,2005-08-31,19.75,0.17,186712.8,1.793754
4,2005-09-30,19.62,0.35,184473.1,1.903378


In [20]:
df['ipca_index'] = (1 + df['ipca']/100).cumprod()

df_vecm = df[[
    'selic',
    'ipca_index',
    'ibc_br',
    'preco_gasolina'
]].dropna()

In [22]:
df_vecm.head(5)

,selic,ipca_index,ibc_br,preco_gasolina
0,19.60,1.004900,178129.5,1.814208
1,19.75,1.004699,179509.8,1.775512
2,19.75,1.007211,181673.2,1.785572
3,19.75,1.008923,186712.8,1.793754
4,19.62,1.012454,184473.1,1.903378


In [25]:
endog = df_vecm[['ipca_index', 'ibc_br', 'preco_gasolina']]
exog = df_vecm[['selic']]

In [26]:
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron


def test_stationarity(series, col):
    series = series.dropna()
    
    # ADF
    adf = adfuller(series)
    
    # PP
    pp = PhillipsPerron(series)
    
    # KPSS
    kpss_test = kpss(series, regression='c', nlags='auto')
    
    return {
        "Variavel": col,
        "ADF_pvalue": adf[1],
        "PP_pvalue": pp.pvalue,
        "KPSS_pvalue": kpss_test[1]
    }


resultados = []
for col in endog.columns:
    res = test_stationarity(df[col], col)
    resultados.append(res)

frame_final = pd.DataFrame(resultados)
frame_final

C:\Users\Hertz Rafael\AppData\Local\Temp\ipykernel_24120\2436622687.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_test = kpss(series, regression='c', nlags='auto')
C:\Users\Hertz Rafael\AppData\Local\Temp\ipykernel_24120\2436622687.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_test = kpss(series, regression='c', nlags='auto')
C:\Users\Hertz Rafael\AppData\Local\Temp\ipykernel_24120\2436622687.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_test = kpss(series, regression='c', nlags='auto')


,Variavel,ADF_pvalue,PP_pvalue,KPSS_pvalue
0,ipca_index,0.998905,1.000000,0.01
1,ibc_br,0.998990,0.998761,0.01
2,preco_gasolina,0.964342,0.962103,0.01


In [27]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

johansen = coint_johansen(endog, det_order=0, k_ar_diff=1)

print(johansen.lr1)   # estatística trace
print(johansen.cvt)   # valores críticos

[41.18478711 20.80422018  5.09457789]
[[27.0669 29.7961 35.4628]
 [13.4294 15.4943 19.9349]
 [ 2.7055  3.8415  6.6349]]


Em um primeiro resultado, foi possível identificar que, ao rodar johansen, foram encontrados 3 vetores de cointegração. sendo que, como a quantidade de vetores são iguais a quantidade de variaveis, isso significa que a série conjunta é estacionária, mesmo que individualmente sejam nao estacionarias.

In [28]:
from statsmodels.tsa.vector_ar.vecm import VECM

vecm = VECM(
    endog,
    exog=exog,
    k_ar_diff=1,
    coint_rank=2  # ou o que você definiu
)

vecm_fit = vecm.fit()

In [31]:
print(vecm_fit.summary())

Det. terms outside the coint. relation & lagged endog. parameters for equation ipca_index
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
exog1             -5.259e-05   7.56e-05     -0.696      0.487      -0.000    9.55e-05
L1.ipca_index         0.5036      0.061      8.229      0.000       0.384       0.624
L1.ibc_br         -6.738e-08   1.74e-08     -3.863      0.000   -1.02e-07   -3.32e-08
L1.preco_gasolina     0.0072      0.005      1.476      0.140      -0.002       0.017
Det. terms outside the coint. relation & lagged endog. parameters for equation ibc_br
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
exog1              -159.1117    277.670     -0.573      0.567    -703.334     385.111
L1.ipca_index      2.138e+05   2.25e+05      0.951

A estimação do modelo VECM revelou a existência de duas relações de cointegração, indicando a presença de equilíbrios de longo prazo entre as variáveis. A primeira relação evidencia uma associação entre inflação e preço da gasolina, enquanto a segunda relaciona a atividade econômica ao preço da gasolina, sugerindo o papel relevante dos preços de energia na dinâmica macroeconômica.

Os coeficientes de correção de erro indicam que as variáveis inflação e preço da gasolina respondem aos desvios do equilíbrio de longo prazo, ajustando-se ao longo do tempo, enquanto a atividade econômica não apresentou ajuste significativo, sugerindo comportamento mais exógeno no sistema.

No curto prazo, observou-se que a inflação apresenta forte persistência, sendo influenciada pela própria defasagem e pela atividade econômica. A variável preço da gasolina também demonstrou comportamento inercial e sensibilidade à inflação.

A taxa Selic, incluída como variável exógena, não apresentou significância estatística, indicando ausência de efeito contemporâneo relevante sobre as variáveis endógenas no período analisado.